# Oasis Infobyte Data Analytics Internship

## Task 3: Data Cleaning

### Objective

The objective of this task is to clean and transform a deliberately messy cafe sales dataset into a reliable, analysis-ready dataset. The cleaning process includes identifying missing values, duplicate records, inconsistent formatting, incorrect data types, and numerical anomalies.

In [1]:
# ==========================================================
# Import Libraries
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("dirty_cafe_sales.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 1. Initial Data Quality Report

Before cleaning the dataset, we first inspect its structure, missing values, duplicate records, data types, and unusual values. This establishes a baseline that can be compared with the cleaned dataset later.

In [4]:
# ==========================================================
# Initial Data Quality Report
# ==========================================================

quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing %": (df.isnull().mean() * 100).round(2).values,
    "Unique Values": df.nunique().values
})

display(quality_report)

,Column,Data Type,Missing Values,Missing %,Unique Values
0,Transaction ID,object,0,0.00,10000
1,Item,object,333,3.33,10
2,Quantity,object,138,1.38,7
3,Price Per Unit,object,179,1.79,8
4,Total Spent,object,173,1.73,19
5,Payment Method,object,2579,25.79,5
6,Location,object,3265,32.65,4
7,Transaction Date,object,159,1.59,367


In [5]:
# ==========================================================
# Duplicate Check
# ==========================================================

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [6]:
# ==========================================================
# Dataset Information
# ==========================================================

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


### Initial Inspection – Observation

The dataset contains 10,000 transaction records and 8 columns. Several columns contain missing values. The numerical fields are currently stored as object/text data types, while the transaction date is also stored as text. These data-quality issues must be addressed before analysis.

The duplicate check should also be reviewed. If no duplicate rows are present, no rows will be removed because duplicate records should not be artificially created merely to demonstrate a cleaning operation.

In [7]:
# ==========================================================
# Inspect Inconsistent and Invalid Values
# ==========================================================

# Check unique values in important categorical columns
for col in ["Item", "Payment Method", "Location"]:
    print("\n" + "="*50)
    print(f"Unique values in: {col}")
    print("="*50)
    print(df[col].value_counts(dropna=False))

# Check numerical columns stored as text
print("\n" + "="*50)
print("Quantity - sample unique values")
print("="*50)
print(df["Quantity"].value_counts(dropna=False).head(20))

print("\n" + "="*50)
print("Price Per Unit - sample unique values")
print("="*50)
print(df["Price Per Unit"].value_counts(dropna=False).head(20))

print("\n" + "="*50)
print("Total Spent - sample unique values")
print("="*50)
print(df["Total Spent"].value_counts(dropna=False).head(20))

# Check transaction dates
print("\n" + "="*50)
print("Transaction Date - sample values")
print("="*50)
print(df["Transaction Date"].value_counts(dropna=False).head(20))


Unique values in: Item
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

Unique values in: Payment Method
Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

Unique values in: Location
Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

Quantity - sample unique values
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
NaN         138
Name: count, dtype: int64

Price Per Unit - sample unique values
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
NaN         179
UNKNOWN     164
Nam

## 2. Missing Data Handling and Standardisation

The dataset contains both missing values and placeholder values such as `ERROR` and `UNKNOWN`. These placeholders do not represent meaningful information, so they are treated as missing values before cleaning.

For categorical columns, missing values will be replaced with the mode because the most frequently occurring valid category provides a reasonable representation.

For numerical columns, values will first be converted to numeric format. Median imputation will be used because the median is less affected by extreme values.

For transaction dates, invalid values will be converted to missing datetime values and handled using the median valid transaction date.

Categorical values will also be standardised by removing unnecessary spaces and ensuring consistent formatting.

In [8]:
# ==========================================================
# Replace Invalid Placeholder Values
# ==========================================================

# Treat ERROR and UNKNOWN as missing values
df_clean = df.copy()

df_clean = df_clean.replace(["ERROR", "UNKNOWN"], np.nan)

print("ERROR and UNKNOWN values have been converted to missing values.")

ERROR and UNKNOWN values have been converted to missing values.


In [9]:
# ==========================================================
# Standardise Categorical Columns
# ==========================================================

categorical_columns = ["Item", "Payment Method", "Location"]

for col in categorical_columns:
    df_clean[col] = df_clean[col].astype("string").str.strip()

# Standardise text formatting
df_clean["Item"] = df_clean["Item"].str.title()
df_clean["Payment Method"] = df_clean["Payment Method"].str.title()
df_clean["Location"] = df_clean["Location"].str.title()

print("Categorical values standardised successfully.")

Categorical values standardised successfully.


In [10]:
# ==========================================================
# Correct Numerical Data Types
# ==========================================================

numeric_columns = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("Numerical columns converted successfully.")

print(df_clean[numeric_columns].dtypes)

Numerical columns converted successfully.
Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object


In [11]:
# ==========================================================
# Correct Transaction Date Data Type
# ==========================================================

df_clean["Transaction Date"] = pd.to_datetime(
    df_clean["Transaction Date"],
    errors="coerce"
)

print("Transaction Date converted to datetime.")

print(df_clean["Transaction Date"].dtype)

Transaction Date converted to datetime.
datetime64[ns]


In [12]:
# ==========================================================
# Check Missing Values After Standardisation
# ==========================================================

missing_after_standardisation = df_clean.isnull().sum()

print("Missing values after converting ERROR/UNKNOWN:")
display(missing_after_standardisation)

Missing values after converting ERROR/UNKNOWN:


,0
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


In [13]:
# ==========================================================
# Missing Data Handling
# ==========================================================

# Numerical columns → median imputation
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numeric_columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical columns → mode imputation
categorical_columns = ["Item", "Payment Method", "Location"]

for col in categorical_columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Transaction Date → median date imputation
median_date = df_clean["Transaction Date"].dropna().median()
df_clean["Transaction Date"] = df_clean["Transaction Date"].fillna(median_date)

print("Missing values handled successfully.")

Missing values handled successfully.


In [14]:
# Check remaining missing values

print("Remaining missing values:")
display(df_clean.isnull().sum())

Remaining missing values:


,0
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


## 3. Outlier Detection

Outliers are unusually high or low values that may affect statistical analysis and model performance.

The Interquartile Range (IQR) method is used to identify outliers in the numerical columns:

- Quantity
- Price Per Unit
- Total Spent

The IQR is calculated as:

**IQR = Q3 − Q1**

Values below **Q1 − 1.5 × IQR** or above **Q3 + 1.5 × IQR** are considered potential outliers.

The outliers will be reviewed before deciding whether they should be retained, capped, or removed.

In [15]:
# ==========================================================
# Outlier Detection using IQR
# ==========================================================

numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

outlier_summary = []

for col in numeric_columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[col] < lower_bound) |
        (df_clean[col] > upper_bound)
    ]

    outlier_summary.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outliers)
    })

outlier_report = pd.DataFrame(outlier_summary)

display(outlier_report)

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Quantity,2.0,4.0,2.0,-1.0,7.0,0
1,Price Per Unit,2.0,4.0,2.0,-1.0,7.0,0
2,Total Spent,4.0,12.0,8.0,-8.0,24.0,259


### Outlier Detection – Observation

The IQR analysis identified no outliers in Quantity or Price Per Unit. However, 259 potential outliers were detected in Total Spent, with values above 24 considered unusually high based on the IQR rule.

These observations were retained because a high transaction amount may represent a legitimate purchase rather than a data-entry error. Removing valid high-value transactions could distort the analysis. Therefore, the identified Total Spent outliers will be retained and documented rather than automatically removed.

## 4. Before vs. After Data Quality Comparison

A before-and-after comparison is created to evaluate the effectiveness of the cleaning process.

The comparison includes:

- Number of rows
- Number of columns
- Total missing values
- Duplicate rows
- Data type accuracy

This provides a clear record of the changes made during data cleaning.

In [16]:
# ==========================================================
# Before vs After Data Quality Summary
# ==========================================================

before_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before Cleaning": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})

after_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "After Cleaning": [
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum()
    ]
})

comparison = before_summary.merge(after_summary, on="Metric")

display(comparison)

,Metric,Before Cleaning,After Cleaning
0,Rows,10000,10000
1,Columns,8,8
2,Missing Values,6826,0
3,Duplicate Rows,0,0


### Before vs. After – Observation

The cleaning process successfully removed missing values by applying appropriate imputation strategies while preserving the original number of records. No duplicate rows were identified in the original dataset, so no records were removed for duplication.

The cleaned dataset now contains complete values across all columns and is ready for further analysis.

## 5. Final Data Validation

After completing the cleaning process, the cleaned dataset is validated to ensure that no missing values remain, numerical columns have the correct data types, the transaction date is stored correctly, and duplicate records are absent.

This final validation confirms that the dataset is ready for analysis.

In [17]:
# ==========================================================
# Final Data Validation
# ==========================================================

print("Final Dataset Shape:", df_clean.shape)

print("\nMissing Values:")
display(df_clean.isnull().sum())

print("\nDuplicate Rows:", df_clean.duplicated().sum())

print("\nFinal Data Types:")
display(df_clean.dtypes)

print("\nFirst 5 Cleaned Records:")
display(df_clean.head())

Final Dataset Shape: (10000, 8)

Missing Values:


,0
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0



Duplicate Rows: 0

Final Data Types:


,0
Transaction ID,object
Item,string[python]
Quantity,float64
Price Per Unit,float64
Total Spent,float64
Payment Method,string[python]
Location,string[python]
Transaction Date,datetime64[ns]



First 5 Cleaned Records:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,8.0,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11


## 6. Export Cleaned Dataset

The cleaned dataset is saved as a new CSV file so that it can be reused for further analysis without modifying the original raw dataset.

In [18]:
# ==========================================================
# Save Cleaned Dataset
# ==========================================================

output_file = "cleaned_cafe_sales.csv"

df_clean.to_csv(output_file, index=False)

print(f"Cleaned dataset saved successfully as: {output_file}")

Cleaned dataset saved successfully as: cleaned_cafe_sales.csv


### Final Validation – Observation

The final validation confirms that the cleaned dataset contains no missing values or duplicate rows. Numerical variables have been converted to appropriate numeric data types, and the Transaction Date column has been converted to datetime format. The cleaned dataset has been successfully exported as a new CSV file while preserving the original dataset.

# Conclusion

The data cleaning process successfully transformed the raw cafe sales dataset into a complete and analysis-ready dataset.

### Key Findings

1. The original dataset contained 6,826 missing values, which were successfully handled using appropriate imputation strategies.
2. `ERROR` and `UNKNOWN` values were identified and treated as missing values.
3. Numerical columns were converted from text to appropriate numeric data types, and Transaction Date was converted to datetime.
4. No duplicate records were found in the dataset.
5. IQR analysis identified 259 potential outliers in Total Spent. These were retained because they may represent genuine high-value transactions rather than data-entry errors.

### Business Recommendations

1. Use the cleaned dataset for reliable sales and customer behaviour analysis.
2. Investigate unusually high transaction amounts separately because they may represent valuable high-spending customers.
3. Maintain standardized payment-method and location records in future data collection to reduce missing and inconsistent data.
4. Apply automated data-validation rules when new transactions are entered to prevent ERROR, UNKNOWN, and invalid values.

In [19]:
# Save final cleaned dataset
df_clean.to_csv("cleaned_cafe_sales.csv", index=False)

print("✅ Task 3 completed successfully!")
print("✅ Cleaned file: cleaned_cafe_sales.csv")

✅ Task 3 completed successfully!
✅ Cleaned file: cleaned_cafe_sales.csv


## Final Status

**Task 3 – Cleaning Data: COMPLETED ✅**

The dataset was inspected, cleaned, standardized, validated, and exported as a new CSV file without modifying the original raw dataset.